In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)



Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive/Labeled Dataset/dataset"




 AI_01	 AI_07	   CD1_075   CD1_093   CD1_107	 CD1_124
 AI_02	 CD1_065   CD1_078   CD1_100   CD1_112	 CD1_91
 AI_03	 CD1_066   CD1_079   CD1_101   CD1_118	'DPSD1 003'
 AI_04	 CD1_067   CD1_082   CD1_104   CD1_122	'DPSD1 007'
 AI_06	 CD1_068   CD1_087   CD1_106   CD1_123


In [ ]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments


In [ ]:
import os
import pandas as pd

csv_dir = "/content/drive/MyDrive/Labeled Dataset/csv dataset/"
drive_base = "/content/drive/MyDrive/Labeled Dataset/dataset/"
output_path = "/content/drive/MyDrive/Labeled Dataset/cleaned_combined_dataset.csv"

dfs = []

for file in os.listdir(csv_dir):
    if not file.endswith(".csv"):
        continue

    path = os.path.join(csv_dir, file)
    df = pd.read_csv(path)

    # Detect text column
    if "text" in df.columns:
        text_col = "text"
    elif "corrected_text" in df.columns:
        text_col = "corrected_text"
    else:
        continue

    # Detect image column
    if "full_image_path" in df.columns:
        img_col = "full_image_path"
    elif "image_path" in df.columns:
        img_col = "image_path"
    else:
        continue

    # Keep only relevant columns
    df = df[["line_id", text_col, img_col]].rename(columns={text_col: "text", img_col: "image_path"})

    # Remove empty/placeholder text
    df = df[~df["text"].isin(["", "0 0", "0 0 "])]

    # Function to safely fix path
    def safe_fix_path(row):
        p = str(row["image_path"]).replace("\\", "/")  # convert backslashes to slashes
        p = p.replace("C:/Users/pavit/Desktop", drive_base)
        p = p.replace("/home/appuser/Desktop", drive_base)

        # Extract folder name from line_id
        folder = row["line_id"].split(".pdf")[0] if pd.notna(row["line_id"]) else "UNKNOWN"
        fname = os.path.basename(p) if p else "UNKNOWN_FILE"
        new_path = os.path.join(drive_base, folder, fname)
        return new_path

    # Apply row-wise
    df["image_path"] = df.apply(safe_fix_path, axis=1)

    dfs.append(df)

# Merge all CSVs
combined_df = pd.concat(dfs, ignore_index=True)
combined_df.drop_duplicates(subset=["line_id"], inplace=True)
combined_df.reset_index(drop=True, inplace=True)

# Check how many images actually exist
existing = combined_df["image_path"].apply(os.path.exists)
print(f"Total rows: {len(combined_df)}, Existing images: {existing.sum()}, Missing: {len(combined_df) - existing.sum()}")

# Save cleaned dataset
combined_df.to_csv(output_path, index=False)
print(f"✅ Clean dataset saved at: {output_path}")


Total rows: 404, Existing images: 355, Missing: 49
✅ Clean dataset saved at: /content/drive/MyDrive/Labeled Dataset/cleaned_combined_dataset.csv


In [ ]:
missing = combined_df[~combined_df["image_path"].apply(os.path.exists)]
for p in missing["image_path"].head(10):
    print(repr(p))


'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page005_line001.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page007_line003.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page012_line002.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page012_line003.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page012_line004.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page012_line006.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page012_line007.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page012_line008.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page013_line001.png'
'/content/drive/MyDrive/Labeled Dataset/dataset/CD1_094/CD1_094.pdf_page013_line002.png'


**1️⃣ Install & Import**

In [ ]:
!pip install transformers datasets evaluate torch torchvision pillow --quiet

import os
import torch
import pandas as pd
from PIL import Image
from datasets import Dataset
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
import evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


**2️⃣ Load the cleaned dataset**

In [ ]:
clean_csv = "/content/drive/MyDrive/Labeled Dataset/cleaned_combined_dataset.csv"
df = pd.read_csv(clean_csv)

# Keep only rows where images exist
df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)

# Split train/validation (approx 90-10 split)
train_df = df.sample(frac=0.9, random_state=42)
val_df = df.drop(train_df.index)

print(f"Train samples: {len(train_df)}, Validation samples: {len(val_df)}")

# Convert to HuggingFace Dataset
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)


Train samples: 320, Validation samples: 35


**3️⃣ Load Processor & Model**

In [ ]:
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

# Set decoder parameters
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# Optional: Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (i

**4️⃣ Preprocessing Function**

In [ ]:
max_target_length = 128

from transformers import TrOCRProcessor
from PIL import Image
import torch

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")

def preprocess(example):
    # Load image
    image = Image.open(example["image_path"]).convert("RGB")

    # Encode image only
    pixel_values = processor.feature_extractor(images=image, return_tensors="pt").pixel_values[0]

    # Encode target text separately
    text = str(example["text"]) if example["text"] is not None else ""
    labels = processor.tokenizer(text, padding="max_length", truncation=True, max_length=128, return_tensors="pt").input_ids[0]

    # Replace pad token with -100 (ignore padding in loss)
    labels[labels == processor.tokenizer.pad_token_id] = -100

    return {
        "pixel_values": pixel_values,
        "labels": labels
    }





# Apply preprocessing
train_ds = train_ds.map(preprocess)
val_ds = val_ds.map(preprocess)

train_ds.set_format(type="torch", columns=["pixel_values", "labels"])
val_ds.set_format(type="torch", columns=["pixel_values", "labels"])


Map:   0%|          | 0/320 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/trocr/processing_trocr.py:139: FutureWarning: `feature_extractor` is deprecated and will be removed in v5. Use `image_processor` instead.
  warnings.warn(


Map:   0%|          | 0/35 [00:00<?, ? examples/s]

**5️⃣ Data Collecctor**

In [ ]:
!pip install jiwer



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.2 MB/s eta 0:00:00


In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence

def data_collator(batch):
    pixel_values = [example["pixel_values"] for example in batch]
    labels = [example["labels"] for example in batch]

    # Stack pixel_values manually
    pixel_values = torch.stack(pixel_values)

    # Pad labels to max length in batch
    labels = pad_sequence([torch.tensor(l) for l in labels], batch_first=True, padding_value=processor.tokenizer.pad_token_id)

    return {"pixel_values": pixel_values, "labels": labels}



**2️⃣ Metric (CER)**

In [ ]:
import evaluate

cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    # Decode predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    # Decode labels
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}


**3️⃣ Training Arguments**

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./trocr_model",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_strategy="no",  # disable logging
    save_total_limit=2,
    num_train_epochs=3,
)



Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


**4️⃣ Trainer**

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    tokenizer=processor,  # works in v4.5.7
    compute_metrics=compute_metrics
)


/tmp/ipython-input-4279257113.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


**5️⃣ Start Training**

In [ ]:
from transformers import Trainer, TrainingArguments
import os

# Disable W&B logging
os.environ["WANDB_DISABLED"] = "true"



In [ ]:
trainer.train()


/tmp/ipython-input-2067349078.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = pad_sequence([torch.tensor(l) for l in labels], batch_first=True, padding_value=processor.tokenizer.pad_token_id)


Step,Training Loss


In [ ]:
# Save the model
model.save_pretrained("./trocr_model_final")

# Save tokenizer separately
processor.tokenizer.save_pretrained("./trocr_model_final/tokenizer")

# Save feature extractor separately
processor.feature_extractor.save_pretrained("./trocr_model_final/feature_extractor")


/usr/local/lib/python3.12/dist-packages/transformers/models/trocr/processing_trocr.py:139: FutureWarning: `feature_extractor` is deprecated and will be removed in v5. Use `image_processor` instead.
  warnings.warn(


['./trocr_model_final/feature_extractor/preprocessor_config.json']

**Evaluate the Model**

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# Reload processor
processor_reloaded = TrOCRProcessor.from_pretrained("./trocr_model")

# Reload model
model_reloaded = VisionEncoderDecoderModel.from_pretrained("./trocr_model")

print("Processor and model loaded successfully!")


OSError: Can't load image processor for './trocr_model'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure './trocr_model' is the correct path to a directory containing a preprocessor_config.json file